## PyTorch Computer Vision 

**torchvision** --> Contains datasets, model architectures and image transformations often used for computer vision problems.

**torchvision.datasets** --> Here you'll find many example computer vision datasets for a range of problems from image classification, object detection, image captioning, video classification and more. It also contains a series of base classes for making custom datasets.

**torchvision.models** -->	This module contains well-performing and commonly used computer vision model architectures implemented in PyTorch, you can use these with your own problems.

**torchvision.transforms** -->	Often images need to be transformed (turned into numbers/processed/augmented) before being used with a model, common image transformations are found here.

**torch.utils.data.Dataset** --> Base dataset class for PyTorch.
**torch.utils.data.DataLoader** -->	Creates a Python iterable over a dataset (created with torch.utils.data.Dataset).

**ToTensor** --> Converts a PIL Image or numpy.ndarray (H x W x C) in the range [0, 255] to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0] if the PIL Image belongs to one of the modes (L, LA, P, I, F, RGB, YCbCr, RGBA, CMYK, 1) or if the numpy.ndarray has dtype = np.uint8

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from torchmetrics import ConfusionMatrix

from mlxtend.plotting import plot_confusion_matrix

import torchvision
from torchvision import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor


from helper_functions import accuracy_fn


import matplotlib.pyplot as plt 
import pandas as pd
import requests

from tqdm.auto import tqdm ## for progress bar

from pathlib import Path

print(torch.__version__)
print(torchvision.__version__)

In [ ]:
## Download helper functions from Learn PyTorch repo

if Path("helper_functions.py").is_file():
    print("helper_functions.py already exists, skipping download...")

else:
    print("Download helper_functions.py")

    request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/refs/heads/main/helper_functions.py")

    with open("helper_functions.py", "wb") as f:
        f.write(requests.content)

### Getting a Dataset

The dataset we'll be using is FashionMINIST from `torchvision.datasets`



In [ ]:
train_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform= ToTensor(),
    target_transform=None
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
    target_transform=None
)

In [ ]:
len(train_data), len(test_data)

In [ ]:
class_name = test_data.classes

print("Class\n", class_name)



In [ ]:
## Check Image Shape 
image , lable = train_data[0]
print("Image Shape: ", image.shape)
print("Image labels: ", class_name[lable])

### Visualizing Data 



In [ ]:
image, lable = train_data[0]
plt.imshow(image.squeeze(), cmap="gray")
plt.title(class_name[lable])
plt.axis(False)

In [ ]:
# Visualize More Image


torch.manual_seed(42)
fig = plt.figure(figsize=(9,9))

rows, cols = 4, 4

for i in range(1, rows*cols+1):
    random_index = torch.randint(0, len(test_data), size=[1]).item()
    image, label= train_data[random_index]

    fig.add_subplot(rows, cols, i)

    plt.imshow(image.squeeze(), cmap="gray")

    plt.title(f"Class: {class_name[label]}")

    plt.axis(False);

In [ ]:
### Now Visualize without random seed.

fig = plt.figure(figsize=(10, 10))

rows, cols = 5, 5

for i in range(1, rows*cols+1):
    random_idx = torch.randint(0, len(train_data), size=[1]).item()

    img, lab = train_data[random_idx]

    fig.add_subplot(rows, cols, i)

    plt.imshow(img.squeeze(), cmap="gray")

    plt.title(class_name[lab])

    plt.axis(False);

In [ ]:
train_data, test_data

In [ ]:
train_loader = DataLoader(dataset=train_data,
                          batch_size=32,
                          shuffle=True)

test_loader = DataLoader(dataset=test_data,
                         batch_size=32,
                         shuffle=False)


In [ ]:
## Let's check out what what we've created 

print(f"DataLoader: {train_loader, test_loader}")

print(f"Length of train dataloader: {len(train_loader)}")
print(f"Length of test dataloader: {len(test_loader)}")

In [ ]:
## Check out what's inside the training dataloader

train_features_batch, train_labels_batch = next(iter(train_loader))

train_features_batch.shape, train_labels_batch.shape

In [ ]:
# show a sample 

torch.manual_seed(42)

random_id = torch.randint(0 , len(train_features_batch), size=[1]).item()

img, labels = train_features_batch[random_id], train_labels_batch[random_id]

plt.imshow(img.squeeze(), cmap="gray")

plt.title(labels)

plt.axis(False);

print(f"Image size: {img.shape}")
print(f"Label: {class_name[labels]}")

### Create Model Using nn.Module and nn.Sequential

### Model 0: Build a basline model

##### When starting to build a series of mahine learning modelling experiments, it's best practice to start with a baseline model. A baseline model is a simple model you will try and improve upon with subsequent models/experiments.
##### In other words: start simply and add complexity when necessary.

In [ ]:
# Create a flatten layer 

flatten_model = nn.Flatten()

# get a single sample 

x = train_features_batch[0]

# Flatten the sample 

output = flatten_model(x) # perform forward pass

print(f"Shape before flattening: {x.shape} -> [color_channels, height, width]")
print(f"Shape after flattening: {output.shape} -> [color_channels, height*width]")

In [ ]:
class FashionMINISTModelV0(nn.Module):
    def __init__(self, 
                 input_shape: int,
                 hidden_units: int,
                 output_shape: int):
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(),

            nn.Linear(in_features=input_shape,
                      out_features=hidden_units),

            nn.Linear(hidden_units, output_shape)

        )

    def forward (self, x):
        return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)

model_0 = FashionMINISTModelV0(
    input_shape=784, # 28*28
    hidden_units= 10,
    output_shape=len(class_name)
   
)

In [ ]:
# Calculate Accuracy 

# def accuracy_fn(y_true= None, y_preds= None):
#     correct = torch.eq(y_true, y_preds).sum().item()

#     acc = (correct/len(y_preds)) * 100 

#     return acc

In [ ]:
lo_fn = nn.CrossEntropyLoss()

op = optim.Adam(params=model_0.parameters(),
                lr=0.001)



## Creating a function to time our experiments 

Machine learning is very experimental.

Two of the main things will often want to track are:
1. Model's performance(loss and accuracy values etc)
2. How fast it runs 

In [ ]:
next(model_0.parameters()).device

In [ ]:
from timeit import default_timer  as timer 

def print_train_time(start: float,
                     end: float ,
                     device: torch.device = None):
    """Prints difference between start and end time."""

    total_time =  end - start

    print(f"Train time on {device}: {total_time:.3f} seconds")

    return total_time

In [ ]:
start_time = timer()

# some code ....

end_time = timer()

print_train_time(start=start_time, end=end_time, device="cpu")

#### Creating a training loop and training the data in batches.

In [ ]:
torch.manual_seed(42)

train_time_start =  timer()


# set the number of epochs 

epochs = 3

# create training and test loop 

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n_____")

    ## Training 

    train_loss = 0

    # Add a loop to loop through the training batches 

    for batch, (x, y) in enumerate(train_loader):
        model_0.train()

        y_pred = model_0(x)

        # calculate loss per batch
        loss =  lo_fn(y_pred, y)

        train_loss += loss

        op.zero_grad()

        loss.backward()

        op.step()

        if batch % 400 == 0:
            print(f"Looked at {batch * len(x)}/{len(train_loader.dataset)} samples.")


    train_loss /= len(train_loader)


#### Testing 

test_loss, test_acc = 0, 0

model_0.eval()

with torch.inference_mode():
    for X_test, Y_test in test_loader:
        test_pred =  model_0(X_test)

        loss = lo_fn(test_pred, Y_test)

        test_loss += loss

        # calculate accuracy 

        test_acc += accuracy_fn(y_true=Y_test, y_pred=test_pred.argmax(dim=1))

    test_loss /= len(test_loader)

    test_acc /= len(test_loader)


print(f"\nTrain loss: {train_loss: .2f} || Test loss: {test_loss: .2f} || Test Accuracy: {test_acc}")

train_end_time = timer()

total_train_time_model0 = print_train_time(start=train_time_start, end=train_end_time,
                                    device=str(next(model_0.parameters()).device))

      



## Make predictions and get Model 0 Results

In [ ]:
torch.manual_seed(42)

def eval_model(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn = torch.nn.Module,
               acc = accuracy_fn,
               device=None):
    """Returns a dictionary containing the results of model predicting on data_loader.."""

    Test_loss, acc = 0, 0

    model.eval()
    with torch.inference_mode():
        for X, Y in tqdm(data_loader):
            X = X.to(device)
            Y = Y.to(device)
            Y_pred = model(X)

            loss = loss_fn(Y_pred, Y)

            Test_loss += loss

            acc += accuracy_fn(y_true=Y,
                               y_pred=Y_pred.argmax(dim=1))

        Test_loss /= len(data_loader)

        acc /= len(data_loader)

    return {"model_name": model.__class__.__name__,
            "model_loss": Test_loss.item(),
            "ModelAccuracy": acc}


model_0_results = eval_model(model=model_0,
                             data_loader=test_loader,
                             loss_fn= lo_fn,
                             acc=accuracy_fn)

model_0_results

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device: ", device)

### Model 1: Building a better model with non-linearity

In [ ]:
class FashionMINISTModelV1(nn.Module):
    def __init__(self, input: int,
                  hidden_units: int, 
                  output: int):
        super().__init__()

        self.layer_stack = nn.Sequential(
            nn.Flatten(), 
            nn.Linear(in_features=input, out_features=hidden_units),
            nn.ReLU(),

            nn.Linear(hidden_units, hidden_units*2),
            nn.ReLU(),

            nn.Linear(hidden_units*2, output)

        )

    def forward (self, x: torch.Tensor):
        return self.layer_stack(x)

In [ ]:
torch.manual_seed(42)

model_1 = FashionMINISTModelV1(
    input=784,
    hidden_units=10, 
    output=len(class_name)
).to(device)

los_fn = nn.CrossEntropyLoss()
opt = optim.Adam(params=model_1.parameters(), 
                 lr = 0.001)

#### Functionizing Training and Testing/evaluation loops

In [ ]:


def train_step(model: torch.nn.Module,
               data_loader: torch.utils.data.DataLoader,
               loss_fn: torch.nn.Module,
               optimizer: torch.optim.Optimizer,
               acc,
               device: torch.device):

    train_loss, train_acc = 0, 0

    model.to(device)
    model.train()

    for batch, (x, y) in enumerate(data_loader):

        x = x.to(device)
        y = y.to(device)

        y_pred = model(x)

        loss = loss_fn(y_pred, y)

        train_loss += loss.item()

        train_acc += acc(
            y_true=y,
            y_pred=torch.softmax(y_pred, dim=1).argmax(dim=1)
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        if batch % 400 == 0:
            print(
                f"Looked at {batch * data_loader.batch_size}/"
                f"{len(data_loader.dataset)} samples."
            )

    train_loss /= len(data_loader)
    train_acc /= len(data_loader)

  

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.2f}%")

    print({
        "model_name": model.__class__.__name__,
        "model_loss": train_loss,
        "ModelAccuracy": train_acc
    })


torch.manual_seed(42)

def test_step(model: torch.nn.Module,
              data_loader: torch.utils.data.DataLoader,
              loss_fn: torch.nn.Module,
              acc=accuracy_fn,
              device: torch.device=device):

    """Returns a dictionary containing the results of model predicting on data_loader."""

    Test_loss, test_acc = 0, 0

    model.to(device)
    model.eval()

    with torch.inference_mode():
        for X, Y in tqdm(data_loader):

            X = X.to(device)
            Y = Y.to(device)

            Y_pred = model(X)

            loss = loss_fn(Y_pred, Y)

            Test_loss += loss.item()

            test_acc += acc(
                y_true=Y,
                y_pred=Y_pred.argmax(dim=1)
            )

        Test_loss /= len(data_loader)
        test_acc /= len(data_loader)

    print(f"Test Loss: {Test_loss:.4f} | Test Accuracy: {test_acc:.2f}%")

    print({
        "model_name": model.__class__.__name__,
        "model_loss": Test_loss,
        "ModelAccuracy": test_acc
    })

In [ ]:
torch.manual_seed(42)


model_1 = model_1.to(device)

Gpu_start_time = timer()

epochs = 3

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch}\n________")

    train_step(
        model=model_1,
        data_loader=train_loader,
        loss_fn=los_fn,
        optimizer=opt,
        acc=accuracy_fn,
        device=device
    )

    test_step(model=model_1,
             data_loader=test_loader,
            loss_fn= los_fn,
            acc=accuracy_fn)

Gpu_end_time = timer()

total_training_time_model1 = Gpu_end_time - Gpu_start_time

print("Training Time in GPU: ", total_training_time_model1)


    

## Model 2: Building a Convolutional Neural Network(CNN) 

CNN"s are also known ConvNets.

CNN's are known for their capabilities to find patterns in visual data.



In [ ]:
class FashionMINISTModelV2(nn.Module):
    def __init__(self, input: int,
                 hidden_units: int,
                 output: int):

        super().__init__()

        self.ConvModel1 = nn.Sequential(
            nn.Conv2d(
                in_channels=input,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.ConvModel2 = nn.Sequential(
            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),

            nn.Conv2d(
                in_channels=hidden_units,
                out_channels=hidden_units,
                kernel_size=3,
                stride=1,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(
                in_features=hidden_units*7*7,
                out_features=output
            )
        )

    def forward(self, x):
        x = self.ConvModel1(x)
      
        x = self.ConvModel2(x)
     
        x = self.classifier(x)

        return x

In [ ]:
torch.manual_seed(42)

model_2 = FashionMINISTModelV2(
    input=1,
    hidden_units=10,
    output=10
).to(device)

loss_f = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model_2.parameters(),
                    lr = 0.001)


In [ ]:
torch.manual_seed(42)

# Create sample batch of random numbers with same size as image batch
images = torch.randn(size=(32, 3, 64, 64)) # [batch_size, color_channels, height, width]
test_image = images[0] # get a single image for testing
print(f"Image batch shape: {images.shape} -> [batch_size, color_channels, height, width]")
print(f"Single image shape: {test_image.shape} -> [color_channels, height, width]") 
print(f"Single image pixel values:\n{test_image}")

### Train with CNN Model 2

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

Cnn_start_time = timer()

train_lo, train_a = 0, 0

epochs = 3

for epoch in tqdm(range(epochs)):
    print(f"Epoch: {epoch + 1}")

    model_2.train()

    train_step(model=model_2,
                   data_loader=train_loader,
                   loss_fn=loss_f,
                   optimizer=optimizer,
                   acc=accuracy_fn,
                   device=device )

    test_step(model=model_2,
                 data_loader=test_loader,
                loss_fn= los_fn,
                acc=accuracy_fn,
                device=device)

Cnn_end_time = timer()

total_training_time_model2 = print_train_time(start=Cnn_start_time,
                                           end=Cnn_end_time,
                                           device=device)

print(f"Train time in cpu model: {total_training_time_model2}")

In [ ]:
model_2_results = eval_model(model=model_2,
                             data_loader=test_loader,
                             loss_fn=loss_f,
                             acc=accuracy_fn,
                             device = device)

model_2_results

In [ ]:
model_1_results = eval_model(model=model_1,
                             data_loader=test_loader,
                             loss_fn=loss_f,
                             acc=accuracy_fn,
                             device=device)

model_1_results

### Compare model results and training time 



In [ ]:
compare_results  = pd.DataFrame([model_0_results,
                                 model_1_results,
                                 model_2_results])

compare_results

In [ ]:
compare_results["training_time"] = [total_train_time_model0,
                                    total_training_time_model1, 
                                    total_training_time_model2]

compare_results

In [ ]:
compare_results.set_index("model_name") [["ModelAccuracy", "training_time", "model_loss"]].plot(kind="barh")
plt.xlabel("Accuracy (%)")
plt.ylabel("Model")

In [ ]:
def make_predictions(model: torch.nn.Module,
                     data:list,
                     device: torch.device = device):

    pred_probs = []

    model.eval()
    with torch.inference_mode():
        for sample in data:
            sample = torch.unsqueeze(sample, dim=0).to(device=device)

            pred_logit = model(sample)


            pred_prob = torch.softmax(pred_logit.squeeze(), dim=0)

            pred_probs.append(pred_prob.cpu())

    return torch.stack(pred_probs)

In [ ]:
import random 

#random.seed(42)

test_samples =[]

test_labels = []

for sample, label in random.sample(list(test_data), k=9):
    test_samples.append(sample)
    test_labels.append(label)

    test_samples[0].shape

In [ ]:
pred_probs = make_predictions(model=model_2,
                              data=test_samples)

pred_probs

In [ ]:
pred_classes = pred_probs.argmax(dim=1)

pred_classes

In [ ]:
plt.figure(figsize=(14,9))
nrows = 3
ncols = 3

for i, sample in enumerate(test_samples):
    plt.subplot(nrows, ncols, i+1)

    plt.imshow(sample.squeeze(), cmap="gray")

    pred_label = class_name[pred_classes[i]]
    real_label = class_name[test_labels[i]]

    title_text = f"pred: {pred_label} | Truth: {real_label}"

    if pred_label == real_label:
        plt.title(title_text, c="g")

    else:
        plt.title(title_text, c="r")

    plt.axis(False)

### Making  a confusion matrix matrix for Classification problem.

1. Make predictions with our trained model on the test dataset 
2. Make a confusion matrix `torchmetrics.ConfusionMetrics`
3. plot the confusion matrix using `mlxtend.plotting.plot_confusion_matrix()`

In [ ]:
y_preds = []

model_2.eval()
with torch.inference_mode():
    for x_test, y_test in tqdm(test_loader, desc="Making predictions.."):
        x_test, y_test = x_test.to(device), y_test.to(device)

        y_logits = model_2(x_test)
    # Turn predictions from logits -> prediction probabilities -> prediction labels
        y_pred = torch.softmax(y_logits, dim=0).argmax(dim=1)

    # Logits → probabilities → predicted class labels
        # y_pred = torch.softmax(y_logits, dim=1).argmax(dim=1)

        y_preds.append(y_pred.cpu())


#print(y_preds)
y_pred_tensor = torch.cat(y_preds)

y_pred_tensor[:10]
        

In [ ]:
# See if torchmetrics exists, if not, install it
try:
    import torchmetrics, mlxtend
    print(f"mlxtend version: {mlxtend.__version__}")
    assert int(mlxtend.__version__.split(".")[1]) >= 19, "mlxtend verison should be 0.19.0 or higher"
except:
    %pip install -q torchmetrics -U mlxtend 
    import torchmetrics, mlxtend
    print(f"mlxtend version: {mlxtend.__version__}")

## setup  confusion instance and compare predictions to targets 



In [ ]:
confmat = ConfusionMatrix(num_classes=len(class_name), task="multiclass")

confmat_tensor = confmat(preds=y_pred_tensor,
                         target=test_data.targets)


fig, ax = plot_confusion_matrix(
    conf_mat=confmat_tensor.numpy(), #matplotlib likes working with numpy
    class_names= class_name,
    figsize=(10, 7)
)

## Save and load best performing Model

In [ ]:
model_path = Path("models")

model_path.mkdir(parents=True, exist_ok=True)

Model_Name = "03_pytorch_classification_CNN_model.pt"

MODEL_SAVE_PATH = model_path / Model_Name

torch.save(obj= model_2.state_dict(),
           f=MODEL_SAVE_PATH)



In [ ]:
torch.manual_seed(42)

loaded_model2 = FashionMINISTModelV2(input=1,
                                    hidden_units=10,
                                    output=len(class_name))

loaded_model2.load_state_dict(torch.load(f= MODEL_SAVE_PATH))

loaded_model2.to(device=device)

In [ ]:
model_2_results

In [ ]:
loaded_model2_results = eval_model(model=loaded_model2,
                                   data_loader=test_loader,
                                   loss_fn=loss_f,
                                   acc=accuracy_fn,
                                   device=device)

loaded_model2_results